In [ ]:
# Load the cleaned dataset and prepare loan issuance years for temporal analysis.
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
df = pd.read_parquet("../data/interim_clean.parquet")
df["year"] = df.issue_d.dt.year
print(df.shape, df.issue_d.min(), df.issue_d.max())

In [ ]:
# Analyze default rates by loan term and issuance year to identify credit-risk trends.
print("overall default rate:", round(df.defaulted.mean(), 4), "| defaults:", df.defaulted.sum(), "| non-defaults:", (df.defaulted == 0).sum())
print(df.groupby("term").defaulted.agg(["mean", "count"]).round(4))
print(df.groupby(["year", "term"]).defaulted.agg(["mean", "count"]).round(4).unstack())

In [ ]:
g = df.groupby([df.issue_d.dt.to_period("M"), "term"]).defaulted.agg(["mean", "count"]).unstack()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
g["mean"].rolling(3).mean().plot(ax=ax[0], title="Default rate by issue month (3-month average)"); ax[0].set_ylabel("default rate")
g["count"].plot(ax=ax[1], title="Number of loans by issue month"); ax[1].set_ylabel("loans")
plt.tight_layout(); plt.savefig("../reports/target_by_month.png", dpi=120); plt.show()

q = df.groupby([df.issue_d.dt.to_period("Q"), "term"]).defaulted.agg(["mean", "count"]).unstack().round(3)
q["share_60"] = (q[("count", 60)] / (q[("count", 36)] + q[("count", 60)])).round(3)
print(q)

In [ ]:
# Assess loan maturity and identify right-censoring risks in historical default outcomes.
snap = pd.Timestamp("2019-03-01")
print("latest last_pymnt_d:", df.last_pymnt_d.max())
df["maturity"] = pd.to_datetime(df.issue_d.values.astype("datetime64[M]") + df.term.values.astype("int64"))
df["matured"] = df.maturity <= snap
print(df.groupby([df.issue_d.dt.to_period("Q"), "term"]).matured.mean().unstack().round(2))
df["censored_cohort"] = df.issue_d >= "2014-04-01"
print(df.groupby(["term", "censored_cohort"]).defaulted.agg(["mean", "count"]).round(4))